# 2장 2강: 분산분석(ANOVA)과 사후 검정 이론 — 실습문제

## 실습 목표

- 세 집단 이상의 평균을 일원배치 분산분석으로 비교할 수 있다.
- F통계량과 p-value를 이용하여 전체 집단 차이를 판단할 수 있다.
- 집단 간·집단 내 변동으로 ANOVA 표를 구성하고 해석할 수 있다.
- ANOVA가 유의할 때 Tukey HSD 사후 검정을 수행할 수 있다.
- 유의한 집단 쌍과 평균 차이의 크기를 근거로 차이 구조를 설명할 수 있다.

## 실습 환경 / 데이터

- Python
- NumPy, pandas
- scipy.stats
- statsmodels
- `ames_housing(1).csv`

| 컬럼 | 의미 |
|---|---|
| `SalePrice` | 주택 판매가격 |
| `OverallQual` | 주택의 전반적인 품질 점수 |
| `KitchenQual` | 주방 품질 |

> 모든 검정의 유의수준은 `α = 0.05`입니다.  
> 표본은 지정된 `random_state`로 추출하여 결과를 재현합니다.

In [ ]:
# ANOVA(일원 분산 분석) : 하나의 집단 구분 기준에 따라 나뉜 여러 집단의 모평균이 모두 같은지 검정하는 방법
# -> 교육방법 a,b,c로 교육을 진행한 뒤 점수를 비교함
# F 통계량 : f통계량은 집단 간 평균 제곱을 집단 내 평균 제곱으로 나눈 값
# -> 귀무가설과 가정이 맞을 때, 두 평균제곱은 같은 분산을 수정
# -> f값이 클수록 집단 간 차이가 내부 퍼짐에 비해 큼

## 실습 준비

1. 필요한 라이브러리를 불러오세요.
2. Ames Housing 데이터를 `df`에 불러오세요.
3. 데이터 크기, 결측치 수, 컬럼명과 상위 5개 행을 확인하세요.

In [1]:
# 실습 준비 코드를 작성하세요.

# 1. 필요한 라이브러리 불러오기
import pandas as pd
from scipy import stats

# 2. Ames Housing 데이터를 df에 불러오기
df = pd.read_csv('ames_housing.csv')

print(df.isna().sum())

# 3. 데이터의 행과 열 개수, 컬럼명, 상위 5개 행 확인
print("행, 열 개수:", df.shape)
print()
print("컬럼명:", df.columns.tolist())
print()
print("상위 5개 행:")
print(df.head())


SalePrice       0
GrLivArea       0
LotArea         0
OverallQual     0
KitchenQual     0
CentralAir      0
HeatingQC       0
PavedDrive      0
Neighborhood    0
YearBuilt       0
dtype: int64
행, 열 개수: (1460, 10)

컬럼명: ['SalePrice', 'GrLivArea', 'LotArea', 'OverallQual', 'KitchenQual', 'CentralAir', 'HeatingQC', 'PavedDrive', 'Neighborhood', 'YearBuilt']

상위 5개 행:
   SalePrice  GrLivArea  LotArea  OverallQual KitchenQual CentralAir  \
0     208500       1710     8450            7          Gd          Y   
1     181500       1262     9600            6          TA          Y   
2     223500       1786    11250            7          Gd          Y   
3     140000       1717     9550            7          Gd          Y   
4     250000       2198    14260            8          Gd          Y   

  HeatingQC PavedDrive Neighborhood  YearBuilt  
0        Ex          Y      CollgCr       2003  
1        Ex          Y      Veenker       1976  
2        Ex          Y      CollgCr       2001  
3   

---

## 필수 1. One-way ANOVA로 전체 평균 차이 확인

### 문제 1-1. 전반적인 품질 점수 5·6·7 집단 비교

#### 문제 설명

주택의 전반적인 품질 점수가 5점, 6점, 7점인 세 집단의 평균 판매가격을 비교하려고 합니다. 각 집단에서 20개씩 표본을 추출하고 일원배치 분산분석을 수행하세요.

#### 요구사항

1. `OverallQual`이 5, 6, 7인 각 집단의 `SalePrice`에서 `n=20`, `random_state=5`로 표본을 추출하세요.
2. 각 집단의 표본 수, 평균, 표준편차를 출력하세요.
3. 세 집단이 서로 다른 주택으로 구성된 독립집단임을 설명하세요.
4. 각 집단에 Shapiro-Wilk 정규성 검정을 수행하세요.
5. 세 집단에 Levene 등분산 검정을 수행하세요.
6. 다음 가설을 작성하세요.
   - H₀: 세 집단의 모집단 평균 판매가격은 모두 같다.
   - H₁: 적어도 한 집단의 모집단 평균 판매가격은 다르다.
7. `stats.f_oneway()`로 일원배치 ANOVA를 수행하세요.
8. F통계량과 p-value를 출력하고 전체 차이 유무를 판단하세요.
9. ANOVA 결과만으로 어느 집단끼리 다른지 알 수 있는지 설명하세요.

#### 해석 질문

**Q1.** 세 집단을 각각 t검정으로 반복 비교하면 어떤 문제가 발생하나요?  
**Q2.** F통계량은 어떤 두 변동의 비율인가요?  
**Q3.** ANOVA 결과 세 집단의 평균 판매가격에는 전체적으로 유의한 차이가 있나요?  
**Q4.** ANOVA 결과만으로 5점·6점·7점 중 어느 집단 쌍이 다른지 알 수 있나요?

#### 제출 결과

- 집단별 기술통계량과 가정 점검 결과
- 가설 설정
- F통계량과 p-value
- 전체 차이 판단
- 사후 검정 필요성 설명
- Q1~Q4 답변

In [2]:
# 필수 1 코드를 작성하세요.

import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv("ames_housing.csv")

# 1. 표본 추출 (n=20, random_state=5)
group_5 = df.loc[df['OverallQual'] == 5, 'SalePrice'].sample(n=20, random_state=5)
group_6 = df.loc[df['OverallQual'] == 6, 'SalePrice'].sample(n=20, random_state=5)
group_7 = df.loc[df['OverallQual'] == 7, 'SalePrice'].sample(n=20, random_state=5)

# 2. 표본 수, 평균, 표준편차 출력
for name, g in [('group_5', group_5), ('group_6', group_6), ('group_7', group_7)]:
    print(f"{name}: 표본 수={len(g)}, 평균={g.mean():.2f}, 표준편차={g.std(ddof=1):.2f}")

# 3. 독립집단 설명
print("\n세 집단은 OverallQual 값이 다른 서로 별개의 주택(행)에서 추출되었습니다.")
print("한 주택은 OverallQual 값을 하나만 가지므로 세 집단 간에 겹치는 대상이 없고, 서로 독립입니다.")

# 4. 정규성 검정
print()
for name, g in [('group_5', group_5), ('group_6', group_6), ('group_7', group_7)]:
    stat, p = stats.shapiro(g)
    print(f"Shapiro-Wilk ({name}): 통계량={stat:.4f}, p-value={p:.4f}")

# 5. 등분산 검정 (Levene은 집단 3개 이상도 한 번에 처리 가능)
stat_levene, p_levene = stats.levene(group_5, group_6, group_7)
print(f"\nLevene 등분산 검정: 통계량={stat_levene:.4f}, p-value={p_levene:.4f}")

# 6. 가설
print("\nH0: 세 집단의 모집단 평균 판매가격은 모두 같다.")
print("H1: 적어도 한 집단의 모집단 평균 판매가격은 다르다.")

# 7. ANOVA
f_stat, p_value = stats.f_oneway(group_5, group_6, group_7)
print(f"\nANOVA 결과: F통계량={f_stat:.4f}, p-value={p_value:.6f}")

# 8. 판단
alpha = 0.05
if p_value < alpha:
    print(f"\np-value({p_value:.6f}) < alpha({alpha}) → 귀무가설 기각")
    print("적어도 한 집단은 평균 판매가격이 다르다고 할 수 있습니다.")
else:
    print(f"\np-value({p_value:.6f}) >= alpha({alpha}) → 귀무가설 기각 실패")
    print("세 집단의 평균 판매가격이 다르다고 할 근거가 부족합니다.")

# 9. ANOVA의 한계
print("\nANOVA는 '세 집단 중 적어도 하나는 다르다'는 것만 알려줄 뿐,")
print("구체적으로 어느 집단과 어느 집단이 다른지는 알려주지 않습니다.")
print("어느 쌍이 다른지 확인하려면 Tukey HSD 같은 사후검정(post-hoc test)을 추가로 수행해야 합니다.")

group_5: 표본 수=20, 평균=130605.00, 표준편차=24937.11
group_6: 표본 수=20, 평균=167826.60, 표준편차=41944.55
group_7: 표본 수=20, 평균=217593.60, 표준편차=48298.39

세 집단은 OverallQual 값이 다른 서로 별개의 주택(행)에서 추출되었습니다.
한 주택은 OverallQual 값을 하나만 가지므로 세 집단 간에 겹치는 대상이 없고, 서로 독립입니다.

Shapiro-Wilk (group_5): 통계량=0.9710, p-value=0.7760
Shapiro-Wilk (group_6): 통계량=0.9527, p-value=0.4096
Shapiro-Wilk (group_7): 통계량=0.9259, p-value=0.1290

Levene 등분산 검정: 통계량=2.6516, p-value=0.0792

H0: 세 집단의 모집단 평균 판매가격은 모두 같다.
H1: 적어도 한 집단의 모집단 평균 판매가격은 다르다.

ANOVA 결과: F통계량=24.2456, p-value=0.000000

p-value(0.000000) < alpha(0.05) → 귀무가설 기각
적어도 한 집단은 평균 판매가격이 다르다고 할 수 있습니다.

ANOVA는 '세 집단 중 적어도 하나는 다르다'는 것만 알려줄 뿐,
구체적으로 어느 집단과 어느 집단이 다른지는 알려주지 않습니다.
어느 쌍이 다른지 확인하려면 Tukey HSD 같은 사후검정(post-hoc test)을 추가로 수행해야 합니다.


### 필수 1 답변 작성란

**Q1.** 세 집단을 각각 t검정으로 반복 비교하면 어떤 문제가 발생하나요?  
-> 검정을 반복할수록 전체분석에서 한 번 이상 제 1종 오류가 발생 확률이 0.05보다 커지는 다중비교 문제가 발생

**Q2.** F통계량은 어떤 두 변동의 비율인가요?  
-> 집단간 평균 차이를 나타내는 집단 간 변동을 같은 집단 내부의 개인차인 집단 내 변동으로 나눈 비율

**Q3.** ANOVA 결과 세 집단의 평균 판매가격에는 전체적으로 유의한 차이가 있나요?  
-> 있음 따라서 적어도 한 집단의 모집단 평균 판매가격은 다르다

**Q4.** ANOVA 결과만으로 5점·6점·7점 중 어느 집단 쌍이 다른지 알 수 있나요?
-> 알 수 없음, 구체적인 집단 쌍은 TUKEY HSD와 같은 사후 검정으로 확인해야함

---

## 필수 2. ANOVA 표 구성과 Tukey HSD 사후 검정

### 문제 2-1. 품질 점수별 차이 구조 확인

#### 문제 설명

필수 1의 세 집단을 이용하여 ANOVA 표를 직접 구성하고, 전체 차이가 유의한 경우 Tukey HSD 사후 검정을 수행해 어느 품질 점수 집단끼리 차이가 있는지 확인하세요.

#### 요구사항

1. 필수 1의 세 집단을 하나의 `anova_df` 데이터프레임으로 결합하세요.
2. 전체 평균을 계산하세요.
3. 다음 값을 계산하여 ANOVA 표를 만드세요.
   - 집단 간 제곱합 `SS_between`
   - 집단 내 제곱합 `SS_within`
   - 집단 간·집단 내 자유도
   - 평균제곱 `MS_between`, `MS_within`
   - F통계량
4. 직접 계산한 F통계량이 `stats.f_oneway()` 결과와 일치하는지 확인하세요.
5. ANOVA p-value가 0.05보다 작을 때만 `pairwise_tukeyhsd()`를 실행하세요.
6. Tukey 결과에서 `reject=True`인 집단 쌍을 확인하세요.
7. 각 집단 평균을 이용하여 집단 쌍별 평균 차이를 계산하세요.
8. 어느 집단 쌍이 유의하며 차이가 가장 큰 집단 쌍은 무엇인지 해석하세요.

#### 해석 질문

**Q1.** ANOVA 표에서 `SS_between`과 `SS_within`은 각각 무엇을 의미하나요?  
**Q2.** Tukey HSD의 `reject=True`는 무엇을 의미하나요?  
**Q3.** 어느 품질 점수 집단 쌍에서 유의한 차이가 확인되나요?  
**Q4.** 평균 판매가격 차이가 가장 큰 집단 쌍은 무엇이며 차이는 얼마인가요?

#### 제출 결과

- ANOVA 표
- F통계량 대조 결과
- Tukey HSD 결과
- 유의한 집단 쌍
- 집단 쌍별 평균 차이
- Q1~Q4 답변

In [5]:
# 필수 2 코드를 작성하세요.
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# (필수1에서 만든 group_5, group_6, group_7이 이미 있다고 가정)

# 1. 세 집단을 하나의 데이터프레임으로 결합
anova_df = pd.DataFrame({
    'SalePrice': pd.concat([group_5, group_6, group_7], ignore_index=True),
    'OverallQual': ['5']*len(group_5) + ['6']*len(group_6) + ['7']*len(group_7)
})

# 2. 전체 평균
grand_mean = anova_df['SalePrice'].mean()
print(f"전체 평균(grand mean): {grand_mean:.2f}")

# 3. ANOVA 표 구성 요소 계산
groups = [group_5, group_6, group_7]
k = len(groups)              # 집단 수 (3)
N = len(anova_df)            # 전체 표본 수 (60)

# 집단 간 제곱합: 각 집단 평균이 전체 평균에서 얼마나 떨어져 있는지
ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in groups)

# 집단 내 제곱합: 각 집단 안에서 개별값이 그 집단 평균에서 얼마나 떨어져 있는지
ss_within = sum(((g - g.mean())**2).sum() for g in groups)

df_between = k - 1        # 2
df_within = N - k         # 57

ms_between = ss_between / df_between
ms_within = ss_within / df_within

f_manual = ms_between / ms_within

print("=== ANOVA 표 (직접 계산) ===")
print(f"SS_between: {ss_between:.2f}, df_between: {df_between}, MS_between: {ms_between:.2f}")
print(f"SS_within:  {ss_within:.2f}, df_within:  {df_within}, MS_within:  {ms_within:.2f}")
print(f"F통계량(직접계산): {f_manual:.4f}")

# 4. stats.f_oneway()와 비교
f_stat, p_value = stats.f_oneway(group_5, group_6, group_7)
print(f"\nstats.f_oneway() 결과: F={f_stat:.4f}, p-value={p_value:.6f}")
print(f"두 값이 일치하는가: {np.isclose(f_manual, f_stat)}")

# 5. p-value < 0.05일 때만 Tukey HSD 실행
alpha = 0.05
if p_value < alpha:
    print(f"\np-value({p_value:.6f}) < {alpha} → 사후검정(Tukey HSD) 수행")
    tukey_result = pairwise_tukeyhsd(
        endog=anova_df['SalePrice'],
        groups=anova_df['OverallQual'],
        alpha=0.05
    )
    print(tukey_result)

    # 6. reject=True인 쌍 확인
    tukey_df = pd.DataFrame(data=tukey_result._results_table.data[1:],
                             columns=tukey_result._results_table.data[0])
    print("\n유의한(reject=True) 집단 쌍:")
    print(tukey_df[tukey_df['reject'] == True])

    # 7. 집단 쌍별 평균 차이 직접 계산
    means = {'5': group_5.mean(), '6': group_6.mean(), '7': group_7.mean()}
    print("\n각 집단 평균:", means)
    print(f"5 vs 6 차이: {means['6'] - means['5']:.2f}")
    print(f"5 vs 7 차이: {means['7'] - means['5']:.2f}")
    print(f"6 vs 7 차이: {means['7'] - means['6']:.2f}")

    # 8. 해석
    diffs = {
        '5 vs 6': abs(means['6'] - means['5']),
        '5 vs 7': abs(means['7'] - means['5']),
        '6 vs 7': abs(means['7'] - means['6']),
    }
    max_pair = max(diffs, key=diffs.get)
    print(f"\n차이가 가장 큰 집단 쌍: {max_pair} (차이: {diffs[max_pair]:.2f})")
else:
    print(f"\np-value({p_value:.6f}) >= {alpha} → 사후검정을 수행하지 않습니다.")

전체 평균(grand mean): 172008.40
=== ANOVA 표 (직접 계산) ===
SS_between: 76194788836.80, df_between: 2, MS_between: 38097394418.40
SS_within:  89564856787.60, df_within:  57, MS_within:  1571313276.98
F통계량(직접계산): 24.2456

stats.f_oneway() 결과: F=24.2456, p-value=0.000000
두 값이 일치하는가: True

p-value(0.000000) < 0.05 → 사후검정(Tukey HSD) 수행
    Multiple Comparison of Means - Tukey HSD, FWER=0.05    
group1 group2 meandiff p-adj    lower       upper    reject
-----------------------------------------------------------
     5      6  37221.6  0.012  7056.6586  67386.5414   True
     5      7  86988.6    0.0 56823.6586 117153.5414   True
     6      7  49767.0 0.0006 19602.0586  79931.9414   True
-----------------------------------------------------------

유의한(reject=True) 집단 쌍:
  group1 group2  meandiff   p-adj       lower        upper  reject
0      5      6   37221.6  0.0120   7056.6586   67386.5414    True
1      5      7   86988.6  0.0000  56823.6586  117153.5414    True
2      6      7   49767.0  0

### 필수 2 답변 작성란

**Q1.** ANOVA 표에서 `SS_between`과 `SS_within`은 각각 무엇을 의미하나요?  
-> SS_between은 집단 평균들이 전체 평균에서 벗어난 집단 간 변동 값 (집단 간 제곱합)
-> SS_within은 각 관측값이 소속 집단 평균에서 벗어난 집단 내 변동 (집단 내 제곱합)

**Q2.** Tukey HSD의 `reject=True`는 무엇을 의미하나요?  
-> 다중비교 오류를 조정한 뒤에도 해당 두 집단의 평균이 같다는 귀무가설을 기각하겠다는 의미

**Q3.** 어느 품질 점수 집단 쌍에서 유의한 차이가 확인되나요?  
-> 5-6,5-7,6-7의 모든 집단 쌍에서 유의한 차이가 확인됨

**Q4.** 평균 판매가격 차이가 가장 큰 집단 쌍은 무엇이며 차이는 얼마인가요?
-> 품질 5점과 7점 집단이며 평균 차이는 약 86,988달러임


---

## 과제. 주방 품질에 따른 판매가격 차이 분석

### 문제 3-1. 주방 품질 `Ex`·`Gd`·`TA` 집단 비교

#### 문제 설명

주방 품질이 `Ex`, `Gd`, `TA`인 세 집단의 평균 판매가격을 비교합니다. 각 집단에서 20개씩 표본을 추출한 뒤 ANOVA와 Tukey HSD를 순서대로 적용하세요.

> 필수 문제에서 학습한 전체 검정→사후 검정 절차를 새로운 집단 변수에 적용하는 과제입니다.

#### 요구사항

1. `KitchenQual`이 `Ex`, `Gd`, `TA`인 각 집단의 `SalePrice`에서 `n=20`, `random_state=18`로 표본을 추출하세요.
2. 세 집단의 표본 수와 평균을 출력하세요.
3. 정규성과 등분산성을 확인하세요.
4. 일원배치 ANOVA를 수행하고 F통계량과 p-value를 출력하세요.
5. ANOVA가 유의한 경우에만 Tukey HSD 사후 검정을 수행하세요.
6. Tukey 결과에서 유의한 집단 쌍을 확인하세요.
7. 각 집단 쌍의 평균 판매가격 차이를 계산하세요.
8. 어느 집단 쌍의 차이가 가장 큰지 포함하여 주방 품질별 차이 구조를 해석하세요.

#### 해석 질문

**Q1.** ANOVA 결과 세 집단의 평균에는 전체적으로 유의한 차이가 있나요?  
**Q2.** 사후 검정은 어떤 조건에서 수행하나요?  
**Q3.** Tukey HSD에서 유의한 차이가 확인된 집단 쌍은 무엇인가요?  
**Q4.** 평균 판매가격 차이가 가장 큰 집단 쌍은 무엇이며 차이는 얼마인가요?

#### 제출 결과

- 집단별 기술통계량과 가정 점검
- ANOVA 결과
- Tukey HSD 결과
- 유의한 집단 쌍과 평균 차이
- 최종 해석
- Q1~Q4 답변

In [10]:
# 과제 코드를 작성하세요.
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

df = pd.read_csv("ames_housing.csv")

# 1. 표본 추출 (n=20, random_state=18)
group_ex = df.loc[df['KitchenQual'] == 'Ex', 'SalePrice'].sample(n=20, random_state=18)
group_gd = df.loc[df['KitchenQual'] == 'Gd', 'SalePrice'].sample(n=20, random_state=18)
group_ta = df.loc[df['KitchenQual'] == 'TA', 'SalePrice'].sample(n=20, random_state=18)

# 2. 표본 수와 평균 출력
for name, g in [('group_ex', group_ex), ('group_gd', group_gd), ('group_ta', group_ta)]:
    print(f"{name}: 표본 수={len(g)}, 평균={g.mean():.2f}")

# 3. 정규성 & 등분산성 확인
print()
for name, g in [('group_ex', group_ex), ('group_gd', group_gd), ('group_ta', group_ta)]:
    stat, p = stats.shapiro(g)
    print(f"Shapiro-Wilk ({name}): 통계량={stat:.4f}, p-value={p:.8f}")

stat_levene, p_levene = stats.levene(group_ex, group_gd, group_ta)
print(f"\nLevene 등분산 검정: 통계량={stat_levene:.4f}, p-value={p_levene:.4f}")

# 4. ANOVA
f_stat, p_value = stats.f_oneway(group_ex, group_gd, group_ta)
print(f"\nANOVA 결과: F통계량={f_stat:.4f}, p-value={p_value:.4e}")

# 5. 유의할 때만 Tukey HSD
alpha = 0.05
if p_value < alpha:
    print(f"\np-value({p_value:.4e}) < {alpha} → Tukey HSD 사후검정 수행")

    anova_df = pd.DataFrame({
        'SalePrice': pd.concat([group_ex, group_gd, group_ta], ignore_index=True),
        'KitchenQual': ['Ex']*len(group_ex) + ['Gd']*len(group_gd) + ['TA']*len(group_ta)
    })

    tukey_result = pairwise_tukeyhsd(
        endog=anova_df['SalePrice'],
        groups=anova_df['KitchenQual'],
        alpha=0.05
    )
    print(tukey_result)

    # 6. 유의한(reject=True) 쌍 확인
    tukey_df = pd.DataFrame(data=tukey_result._results_table.data[1:],
                             columns=tukey_result._results_table.data[0])
    print("\n유의한(reject=True) 집단 쌍:")
    print(tukey_df[tukey_df['reject'] == True])

    # 7. 집단 쌍별 평균 차이 계산
    means = {'Ex': group_ex.mean(), 'Gd': group_gd.mean(), 'TA': group_ta.mean()}
    print("\n각 집단 평균:", means)
    diffs = {
        'Ex vs Gd': abs(means['Ex'] - means['Gd']),
        'Ex vs TA': abs(means['Ex'] - means['TA']),
        'Gd vs TA': abs(means['Gd'] - means['TA']),
    }
    for pair, d in diffs.items():
        print(f"{pair} 차이: {d:.2f}")

    # 8. 해석
    max_pair = max(diffs, key=diffs.get)
    print(f"\n차이가 가장 큰 집단 쌍: {max_pair} (차이: {diffs[max_pair]:.2f})")
else:
    print(f"\np-value({p_value:.6f}) >= {alpha} → 사후검정을 수행하지 않습니다.")


group_ex: 표본 수=20, 평균=313983.05
group_gd: 표본 수=20, 평균=188835.00
group_ta: 표본 수=20, 평균=137486.60

Shapiro-Wilk (group_ex): 통계량=0.9669, p-value=0.68754074
Shapiro-Wilk (group_gd): 통계량=0.9723, p-value=0.80239455
Shapiro-Wilk (group_ta): 통계량=0.9519, p-value=0.39734081

Levene 등분산 검정: 통계량=2.5550, p-value=0.0866

ANOVA 결과: F통계량=54.8000, p-value=5.3052e-14

p-value(5.3052e-14) < 0.05 → Tukey HSD 사후검정 수행
      Multiple Comparison of Means - Tukey HSD, FWER=0.05       
group1 group2  meandiff  p-adj     lower        upper     reject
----------------------------------------------------------------
    Ex     Gd -125148.05    0.0 -166883.2108  -83412.8892   True
    Ex     TA -176496.45    0.0 -218231.6108 -134761.2892   True
    Gd     TA   -51348.4 0.0122  -93083.5608   -9613.2392   True
----------------------------------------------------------------

유의한(reject=True) 집단 쌍:
  group1 group2   meandiff   p-adj        lower        upper  reject
0     Ex     Gd -125148.05  0.0000 -166883.2108  -83

### 과제 답변 작성란

**Q1.** ANOVA 결과 세 집단의 평균에는 전체적으로 유의한 차이가 있나요?  
-> 있다고 볼 수 있음
-> p=5.3052e-14 으로 0.05보다 작기 때문에 귀무가설을 기각함
-> 주방 품질 세 집단중 적어도 한 집단은 평균 판매 가격의 차이가 있음을 통계적 근거로 볼 수 있음


**Q2.** 사후 검정은 어떤 조건에서 수행하나요?  
-> ANOVA의 P-value값이 0.05보다 작을 때
-> 세 집단중 적어도 한 집단은 평균 판매 가격의 차이가 있음을 통계적 근거로 볼 수 있기 때문에
-> 어떤 집단에서 차이가 발생했는지 확인하기 위해 사후 검정을 수행

**Q3.** Tukey HSD에서 유의한 차이가 확인된 집단 쌍은 무엇인가요?  
-> 세 쌍 모두 0.05보다 작기 때문에 귀무가설을 기각함
-> Ex Gd / Ex TA / Gd TA 세쌍 모두 판매가격 차이가 있다는 것을 통계적 근거로 볼 수 있음


**Q4.** 평균 판매가격 차이가 가장 큰 집단 쌍은 무엇이며 차이는 얼마인가요?
-> Ex vs TA 차이: 176496.45 이므로 가장 큼
-> Ex-TA(176,496.45) > Ex-Gd(125,148.05) > Gd-TA(51,348.40)

---

## 실습 마무리

1. 세 집단 이상을 t검정으로 반복 비교하면 왜 제1종 오류가 커지나요?
-> 각 검정마다 위양성 가능성이 있기 때문에 비교 횟수가 늘어날수록 전체 분석에서 한 번 이상 잘못 기각할 확률이 누적임

2. ANOVA의 귀무가설과 대립가설은 무엇인가요?
-> 귀무가설 : 모든 집단의 모집단과 평균이 같다, 대립가설 : 적어도 한 집단의 평균이 다름

3. F통계량이 크다는 것은 무엇을 의미하나요?
-> 집단 내 변동에 비해 집단 간 평균 차이로 설명되는 변동이 상대적으로 크다는 의미

4. ANOVA가 유의하더라도 사후 검정이 필요한 이유는 무엇인가요?
-> ANOVA는 적어도 한 집단이 다르다는 사실만 알려주며 구체적인 집단 쌍은 알려주지 않음

5. Tukey HSD 결과에서 어떤 항목을 확인해야 하나요?
-> 비교한 집단 쌍, 평균 차이, 조정된 P-VALUE, 신뢰구간, REJECT 여부